In [ ]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Cell 1: Basic paths and result file names
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd()
DATA_INPUT_DIR = PROJECT_ROOT / "data"
DATA_RESULTS_DIR = PROJECT_ROOT / "data_results"
DATA_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CASE_ID = "C0002"
REVISION = "R00"
RESULT_ID = f"{CASE_ID}{REVISION}"

SCENARIO_CODE = "PARK25"
BTMS_CODE = "LC"
MODEL_CODE = "1D"
TASK_CODE = "VAL"
DESIGN_STATUS_CODE = "DES"

RESULT_BASENAME = f"{RESULT_ID}_{SCENARIO_CODE}_{BTMS_CODE}_{MODEL_CODE}_{TASK_CODE}"
RESULT_EXCEL_PATH = DATA_RESULTS_DIR / f"{RESULT_BASENAME}.xlsx"
RESULT_FIGURE_PATH = DATA_RESULTS_DIR / f"{RESULT_BASENAME}.png"
SCRIPT_VERSION = f"{RESULT_BASENAME}.ipynb"

print("Basic paths loaded.")
print(f"Result basename = {RESULT_BASENAME}")
print(f"Result Excel path = {RESULT_EXCEL_PATH}")
print(f"Result figure path = {RESULT_FIGURE_PATH}")


In [ ]:
# ------------------------------------------------------------
# Cell 2: Read Park/Liu single-cell heat-generation data
# ------------------------------------------------------------

def normalize_column_name(name):
    """Normalize Excel column names for robust matching."""
    return str(name).strip().lower().replace(" ", "")


def find_column(dataframe, candidates):
    """Find a column by case/space-insensitive matching."""
    normalized = {normalize_column_name(col): col for col in dataframe.columns}

    for candidate in candidates:
        key = normalize_column_name(candidate)
        if key in normalized:
            return normalized[key]

    raise KeyError(
        f"None of the expected columns were found: {candidates}. "
        f"Existing columns: {list(dataframe.columns)}"
    )


def numeric_series(dataframe, candidates):
    """Return a numeric Series from the first matching candidate column."""
    col = find_column(dataframe, candidates)
    series = pd.to_numeric(dataframe[col], errors="coerce").dropna()
    return col, series


def first_float_or_none(dataframe, candidates):
    """Read the first numeric value from a candidate column if it exists."""
    try:
        _, values = numeric_series(dataframe, candidates)
    except KeyError:
        return None

    if values.empty:
        return None

    return float(values.iloc[0])


HEAT_INPUT_FILE = DATA_INPUT_DIR / "MD05E070207A1  data_power gen_single cell_Liu_20260421.xlsx"
heat_input_df = pd.read_excel(HEAT_INPUT_FILE, sheet_name="Sheet1")

# Geometry is only used for fallback conversion from volumetric heat generation.
H_bat_from_file = first_float_or_none(heat_input_df, ["H(高度)", "H", "height", "Height"])
D_bat_from_file = first_float_or_none(heat_input_df, ["D(直径)", "D", "diameter", "Diameter"])

H_bat_for_q = H_bat_from_file if H_bat_from_file is not None else 65.0e-3
D_bat_for_q = D_bat_from_file if D_bat_from_file is not None else 18.0e-3
V_bat_for_q = np.pi * D_bat_for_q**2 / 4.0 * H_bat_for_q

try:
    # Prefer Qbat(W), because it is already single-cell heat-generation power.
    # Do not use Qpack(W); module heat is assembled inside this model.
    power_col, power_series = numeric_series(
        heat_input_df,
        ["Qbat(W)", "Qbat", "qbat(W)"]
    )
    power_generation_data = power_series.to_numpy(dtype=float)
    power_source_note = f"{power_col} directly, single-cell heat generation"

except KeyError:
    # Fallback: convert volumetric heat generation to single-cell power.
    qvol_col, qvol_series = numeric_series(
        heat_input_df,
        ["qbat(W/m^3)", "qbat(W/m3)", "qbat", "qgen(W/m^3)", "qgen"]
    )
    power_generation_data = qvol_series.to_numpy(dtype=float) * V_bat_for_q
    power_source_note = (
        f"{qvol_col} multiplied by computed cell volume "
        f"{V_bat_for_q:.8e} m3"
    )

dt = 1.0
heat_generation_steps = len(power_generation_data)
num_time_steps = heat_generation_steps + 1

if heat_generation_steps != 1920:
    print(
        f"WARNING: heat-generation data contains {heat_generation_steps} rows, "
        "not 1920 rows."
    )

print(f"Loaded heat-generation data from: {HEAT_INPUT_FILE}")
print(f"Power source: {power_source_note}")
print(
    f"Heat-generation intervals: {heat_generation_steps}; "
    f"temperature states: {num_time_steps}; "
    f"simulation end time: {heat_generation_steps * dt:.0f} s"
)
print(
    f"Q_cell range: min={np.min(power_generation_data):.6g} W, "
    f"max={np.max(power_generation_data):.6g} W, "
    f"mean={np.mean(power_generation_data):.6g} W per cell"
)
print(
    f"Geometry used only for q conversion: "
    f"H={H_bat_for_q:.6g} m, D={D_bat_for_q:.6g} m, "
    f"V={V_bat_for_q:.8e} m3"
)


In [ ]:
# ------------------------------------------------------------
# Cell 3: Battery and module parameters
# Park 20x16 module with one representative 16-cell channel
# ------------------------------------------------------------

T_bat_init = 25.0      # degC, initial battery temperature

m_bat = 48.0e-3        # kg, single-cell mass
cp_bat = 830.0         # J/(kg*K), single-cell specific heat
D_bat = 18.0e-3        # m, cell diameter

# Park battery module layout
N_r, N_c = 16, 20
N_cells_physical = N_r * N_c   # 320 physical cells in the full module

# One representative straight liquid-cooling channel.
# The solver explicitly resolves 16 single-cell thermal nodes, not all 320 cells.
N_cell_per_channel = 16
N_cells = N_cell_per_channel
N_cool_seg = N_cell_per_channel
num_segment = N_cool_seg

assert N_cool_seg == N_cells, "N_cool_seg must equal N_cells in the one-to-one model."
assert N_cells_physical % N_cell_per_channel == 0, (
    "N_cells_physical must be divisible by N_cell_per_channel."
)

# Number of identical representative channels used for full-module heat-rate scaling.
N_base_parallel_channel = N_cells_physical // N_cell_per_channel

print("Battery parameters loaded.")
print(f"Physical cells in full module: {N_cells_physical}")
print(f"Thermal nodes used in solver: {N_cells}")
print(f"Cells per representative channel: {N_cell_per_channel}")
print(f"Base parallel-channel count for 320 cells: {N_base_parallel_channel}")
print(f"m_bat(single cell) = {m_bat:.4f} kg")
print(f"cp_bat(single cell) = {cp_bat:.1f} J/(kg*K)")
print(f"D_bat = {D_bat*1000:.1f} mm")


In [ ]:
# ------------------------------------------------------------
# Cell 4: Liquid-cooling geometry and coolant properties
# ------------------------------------------------------------

# Single physical circular coolant channel
D_channel = 4.0e-3     # m, channel hydraulic diameter
D_h = D_channel
m_dot_liq = 0.0025445  # kg/s, mass flow rate of one physical channel

# One cell is coupled to 6/5 equivalent coolant channels.
# This factor is applied only to the coolant domain, not to battery heat capacity
# or battery heat generation.
fluid_domain_factor = 6.0 / 5.0
m_dot_liq_eff = m_dot_liq * fluid_domain_factor

# Coolant inlet condition
T_liq_in = 25.0        # degC
p_liq = 101325         # Pa
fluid_liq = "Water"

# Constant water properties near 25 degC
rho_liq_const = 998.2  # kg/m3
cp_liq_const = 4182.0  # J/(kg*K)
k_liq_const = 0.6      # W/(m*K)
mu_liq_const = 1.0e-3  # Pa*s

# Aluminum cold plate conduction resistance
k_plate = 202.4        # W/(m*K)
t_plate_total = 12e-3  # m

# Straight-channel length and segmentation
L_channel_total = 0.37971337   # m
L_seg = L_channel_total / N_cool_seg

# Effective fluid-domain area and heat-transfer area
A_channel_cross_single = np.pi * D_channel**2 / 4.0
P_channel_wet_single = np.pi * D_channel

A_channel_cross = A_channel_cross_single * fluid_domain_factor
A_HT_seg = P_channel_wet_single * L_channel_total * fluid_domain_factor / N_cool_seg

# Velocity is calculated for one physical channel.
# Since A_channel_cross is multiplied by 6/5, rho*u*A inside the solver gives
# the effective coolant heat-capacity rate coupled to one cell.
V_cool_in = m_dot_liq / (rho_liq_const * A_channel_cross_single)

# One battery thermal node corresponds to one coolant segment.
cell_to_cool_map = {j: (j,) for j in range(N_cool_seg)}

print("Liquid-cooling parameters loaded.")
print(f"N_cool_seg = {N_cool_seg}")
print(f"L_channel_total = {L_channel_total:.8f} m")
print(f"L_seg = {L_seg:.8f} m")
print(f"D_channel = {D_channel*1000:.1f} mm")
print(f"A_HT_seg = {A_HT_seg:.6e} m2")
print(f"A_channel_cross(effective) = {A_channel_cross:.6e} m2")
print(f"m_dot_liq(single channel) = {m_dot_liq:.7f} kg/s")
print(f"m_dot_liq_eff(per cell fluid domain) = {m_dot_liq_eff:.7f} kg/s")
print(f"Coolant velocity(single physical channel) = {V_cool_in:.3f} m/s")
print(f"Generated cell_to_cool_map: {len(cell_to_cool_map)} one-to-one pairs")


In [ ]:
# ------------------------------------------------------------
# Cell 5: Liquid-side heat transfer coefficient
# Constant 25 degC water properties are used for the Park25 preliminary LC design.
# ------------------------------------------------------------

import lib.BTMS_model as BTMS_model

Pr_liq_25 = cp_liq_const * mu_liq_const / k_liq_const
Re_liq_25 = rho_liq_const * V_cool_in * D_h / mu_liq_const

Nu_liq_25 = BTMS_model.liquid_nusselt_number(
    Re_liq_25,
    Pr_liq_25,
    heating=False
)

htc_liq_25 = Nu_liq_25 * k_liq_const / D_h

# Global heat transfer coefficient including cold-plate conduction resistance.
R_plate = t_plate_total / k_plate
htc_global_25 = 1.0 / (1.0 / htc_liq_25 + R_plate)

# All coolant segments use the same HTC in this reduced model.
htc_seg_const = np.ones(num_segment) * htc_global_25

print("Liquid-side heat transfer coefficient loaded.")
print(f"Pr_liq_25 = {Pr_liq_25:.3f}")
print(f"Re_liq_25(single physical channel) = {Re_liq_25:.1f}")
print(f"Nu_liq_25 = {Nu_liq_25:.3f}")
print(f"htc_liq_25 = {htc_liq_25:.2f} W/(m2*K)")
print(f"R_plate = {R_plate:.6e} m2*K/W")
print(f"htc_global_25 = {htc_global_25:.2f} W/(m2*K)")
print(f"htc_seg_const length = {len(htc_seg_const)}")


In [ ]:
# ------------------------------------------------------------
# Cell 6: Main 1D liquid-cooling solver
# 16 single-cell thermal nodes / 16 coolant segments
# ------------------------------------------------------------

start_time = time.time()

T_bat_history = np.empty(num_time_steps, dtype=object)
T_cool_history = np.empty(num_time_steps, dtype=object)

Q_bat_history = np.zeros(num_time_steps)
Q_cool_history = np.zeros(num_time_steps)
Q_res_history = np.zeros(num_time_steps)

for i in range(num_time_steps):
    T_bat_history[i] = np.ones(N_cells) * T_bat_init
    T_cool_history[i] = np.ones(N_cool_seg) * T_liq_in

debug = False
upwind_scheme = True
progress_interval = 50

print(f"N_cells = {N_cells}", flush=True)
print(f"N_cells_physical = {N_cells_physical}", flush=True)
print(f"N_cool_seg = {N_cool_seg}", flush=True)
print(f"N_base_parallel_channel = {N_base_parallel_channel}", flush=True)
print(f"m_dot_liq(single channel) = {m_dot_liq:.7f} kg/s", flush=True)
print(f"m_dot_liq_eff(per cell fluid domain) = {m_dot_liq_eff:.7f} kg/s", flush=True)
print(f"Total time steps to solve = {num_time_steps - 1}", flush=True)
print(f"Progress will be displayed every {progress_interval} step(s).", flush=True)

for i in range(1, num_time_steps):

    t_current = (i - 1) * dt

    show_progress = (
        i == 1
        or i % progress_interval == 0
        or i == num_time_steps - 1
    )

    if show_progress:
        progress = i / (num_time_steps - 1) * 100
        elapsed = time.time() - start_time
        print(
            f"[START] Step {i}/{num_time_steps - 1}, "
            f"t = {t_current:.0f} s, "
            f"progress = {progress:.2f}%, "
            f"elapsed = {elapsed:.1f} s",
            flush=True
        )

    step_start_time = time.time()

    T_bat_pre = T_bat_history[i - 1]
    T_cool_pre = T_cool_history[i - 1]

    args = {
        "dt": dt,
        "num_seg_bat": N_cells,
        "num_seg_cool": N_cool_seg,

        "T_bat_pre": T_bat_pre,
        "T_cool_pre": T_cool_pre,

        "u_cool_in": V_cool_in,
        "p_cool": p_liq,
        "fluid_cool": fluid_liq,

        "A_HT_seg": A_HT_seg,
        "A_cool_cs": A_channel_cross,

        "m_bat": m_bat,
        "cp_bat": cp_bat,
        "D_bat": D_bat,

        "T_cool_in": T_liq_in,
        "is_cool": True,

        "htc_cool": htc_seg_const,
        "cp_cool": cp_liq_const,
        "rho_cool": rho_liq_const,

        "Q_gen": power_generation_data[i - 1],

        "upwind_scheme": upwind_scheme,
        "cell_to_cool_map": cell_to_cool_map,
    }

    T_dist = BTMS_model.solve_coolant_temperature_distribution(
        args,
        tol=1e-3,
        maxiter=1000,
        debug=debug
    )

    T_cool = T_dist[0:N_cool_seg]
    T_bat = T_dist[N_cool_seg:]

    Q_cool_HT, Q_cool_change, Q_bat = BTMS_model.cal_energy_balance(
        T_dist,
        args
    )

    Q_bat_history[i] = np.sum(Q_bat)
    Q_cool_history[i] = np.sum(Q_cool_HT)

    # Energy residual check for the representative 16-cell channel.
    # There is no cell_group_factor here: one thermal node is one real cell.
    Q_gen_total = power_generation_data[i - 1] * N_cells
    Q_res_history[i] = abs(Q_gen_total - np.sum(Q_bat) - np.sum(Q_cool_HT))

    T_bat_history[i] = T_bat
    T_cool_history[i] = T_cool

    if show_progress:
        step_elapsed = time.time() - step_start_time
        elapsed = time.time() - start_time
        remaining_steps = (num_time_steps - 1) - i
        eta = (elapsed / i) * remaining_steps

        print(
            f"[DONE ] Step {i}/{num_time_steps - 1}, "
            f"step time = {step_elapsed:.2f} s, "
            f"ETA = {eta/60:.1f} min, "
            f"Tmax = {np.max(T_bat):.3f} degC, "
            f"Tmin = {np.min(T_bat):.3f} degC",
            flush=True
        )

end_time = time.time()

print("Transient simulation finished.", flush=True)
print(f"Final simulation time = {(num_time_steps - 1) * dt:.0f} s", flush=True)
print(f"Elapsed wall time = {end_time - start_time:.2f} s", flush=True)
print(f"Max residual = {np.max(Q_res_history):.6e} W", flush=True)


In [ ]:
# ------------------------------------------------------------
# Cell 7: Plot and save maximum/minimum temperatures
# ------------------------------------------------------------

time_s = np.arange(num_time_steps) * dt

# T_bat_history[i] contains 16 single-cell thermal-node temperatures.
Tmax_1D = np.array([
    np.max(T_bat_history[i]) for i in range(num_time_steps)
])

Tmin_1D = np.array([
    np.min(T_bat_history[i]) for i in range(num_time_steps)
])

DeltaT_1D = Tmax_1D - Tmin_1D
Tcool_out_1D = np.array([
    T_cool_history[i][-1] for i in range(num_time_steps)
])

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(time_s, Tmax_1D, linestyle="-", label="T_max")
ax.plot(time_s, Tmin_1D, linestyle="--", label="T_min")

ax.set_xlabel("Simulation time (s)")
ax.set_ylabel("Battery temperature (degC)")
ax.set_title("Temperature response of 16 single-cell thermal nodes")
ax.grid(True)
ax.legend()
fig.tight_layout()
fig.savefig(RESULT_FIGURE_PATH, dpi=300, bbox_inches="tight")
plt.show()

print(f"T_max = {np.max(Tmax_1D):.3f} degC")
print(f"T_min_final = {Tmin_1D[-1]:.3f} degC")
print(f"Maximum temperature difference = {np.max(DeltaT_1D):.3f} degC")
print(f"Maximum coolant outlet temperature = {np.max(Tcool_out_1D):.3f} degC")
print(f"Saved figure: {RESULT_FIGURE_PATH}")


In [ ]:
# ------------------------------------------------------------
# Cell 8: Save standardized result Excel only
# ------------------------------------------------------------

Q_cell_input_W = np.concatenate(([0.0], power_generation_data))
Q_channel_input_W = Q_cell_input_W * N_cells
Q_module_input_W = Q_cell_input_W * N_cells_physical

# The 1D solver result is for one representative 16-cell straight channel.
# Under the identical parallel-channel assumption, heat rates can be scaled
# by N_base_parallel_channel to obtain a full-module equivalent heat rate.
Q_bat_module_equiv_W = Q_bat_history * N_base_parallel_channel
Q_cool_module_equiv_W = Q_cool_history * N_base_parallel_channel

result_df = pd.DataFrame({
    "time_s": time_s,
    "Q_cell_W": Q_cell_input_W,
    "Q_channel_16cell_W": Q_channel_input_W,
    "Q_module_320cell_W": Q_module_input_W,
    "Tmax_1D_C": Tmax_1D,
    "Tmin_1D_C": Tmin_1D,
    "DeltaT_1D_C": DeltaT_1D,
    "Tcool_out_1D_C": Tcool_out_1D,
    "Q_bat_channel_W": Q_bat_history,
    "Q_cool_channel_W": Q_cool_history,
    "Q_bat_module_equiv_W": Q_bat_module_equiv_W,
    "Q_cool_module_equiv_W": Q_cool_module_equiv_W,
    "Q_residual_channel_W": Q_res_history,
})

result_df.to_excel(RESULT_EXCEL_PATH, index=False)

print(f"Saved standardized result Excel: {RESULT_EXCEL_PATH}")
print("Only one table file is generated for this case.")

result_df.head()
